# U-Net from Scratch in PyTorch: Brain MRI Tumor Segmentation

**Day 5 of 10 · Deep Learning Architectures Series**

In this notebook I implement **U-Net** (Ronneberger et al., 2015) from scratch and train it to segment lower-grade glioma regions in brain MRI slices. Segmentation means the output is not one label per image but a **mask with one label per pixel**. Beyond training one model, I run controlled experiments on the three ideas that make segmentation work:

1. **Skip connections**: U-Net vs the same encoder-decoder without skips
2. **The loss function**: BCE vs BCE + Dice, because tumor pixels are a tiny fraction of the image
3. **Data augmentation**

and, building on the lesson from Day 4, a **data-leakage experiment**: a patient-level split vs a random slice-level split, to measure how much leakage inflates the scores.

**What's inside**
1. Background: segmentation, U-Net, Dice/IoU
2. Dataset, patient-level split, and in-memory data pipeline
3. U-Net implementation (with a switch to remove the skips)
4. Five experiments (A to E)
5. Results, qualitative comparison, error analysis, leakage experiment
6. Analysis, limitations, next steps

> Papers: *U-Net: Convolutional Networks for Biomedical Image Segmentation* (Ronneberger, Fischer, Brox, 2015) · *Association of genomic subtypes of lower-grade gliomas with shape features automatically extracted by a deep learning algorithm* (Buda, Saha, Mazurowski, 2019)

> ⚠️ **Research and learning only.** This is an educational project on a public dataset, not a medical device, and the results must not be used for clinical decisions.

> ⚙️ **Kaggle setup:**
> 1. Settings → Accelerator → **GPU (T4 or P100)**. Internet is **not** needed (no pretrained weights).
> 2. On the right panel click **+ Add Input** → search for **"LGG MRI Segmentation"** (by Mateusz Buda) → add it.
> 3. Expected runtime: roughly 30 to 45 minutes for all five experiments.

## 1. Background

### Semantic segmentation
Classification maps an image to one label. **Semantic segmentation** maps every pixel to a class. Here the task is binary: *tumor region (FLAIR abnormality)* vs *background*, for each 256×256 MRI slice.

### U-Net
U-Net is an **encoder-decoder** network with a distinctive shape:

- **Encoder (contracting path):** repeated blocks of two 3×3 convolutions followed by 2×2 max-pooling. Resolution halves and channels double (32 → 64 → 128 → 256 → 512 here). The network sees more context but loses spatial detail.
- **Decoder (expanding path):** transposed convolutions (2×2, stride 2) double the resolution and halve the channels again.
- **Skip connections:** at every resolution, the encoder feature map is **concatenated** with the upsampled decoder feature map. This hands fine spatial detail from the encoder directly to the decoder.
- **Head:** a 1×1 convolution produces one logit per pixel.

In signal-processing terms, pooling is low-pass filtering followed by decimation, so high-frequency detail (sharp boundaries) is lost on the way down. A transposed convolution is zero-insertion upsampling followed by a learned interpolation filter, which cannot recover that lost detail by itself. The skip connections **re-inject the high-frequency detail** from the same resolution of the encoder.

Differences from the original paper: I use `padding=1` ("same") convolutions, so the output has the input's size (the paper used unpadded convolutions and cropping), and BatchNorm after every convolution (not in the original).

### Metrics
For a predicted mask $P$ and a ground-truth mask $G$:

$$\mathrm{Dice} = \frac{2\,|P \cap G|}{|P| + |G|}, \qquad \mathrm{IoU} = \frac{|P \cap G|}{|P \cup G|}, \qquad \mathrm{Dice} = \frac{2\,\mathrm{IoU}}{1 + \mathrm{IoU}}$$

Pixel accuracy is misleading here: most pixels are background, so predicting "no tumor" everywhere already scores very high. I report:
- **Global Dice and IoU:** computed from the sums over all test pixels
- **Mean Dice / IoU on tumor slices:** the per-slice average over slices that contain a tumor
- **Slice-level sensitivity and specificity:** does the model correctly flag slices with / without a tumor (a prediction counts as "tumor" if it has at least 10 pixels)

### The loss and class imbalance
Binary cross-entropy treats every pixel equally, so with few tumor pixels the easy solution is to predict background. The **soft Dice loss** directly optimizes overlap:

$$\mathcal{L}_{Dice} = 1 - \frac{2\sum p_i g_i + \epsilon}{\sum p_i + \sum g_i + \epsilon}$$

computed over the whole batch (so empty slices do not cause 0/0). I compare BCE alone with BCE + Dice.

### Data leakage in medical imaging
The dataset contains many **slices from the same patient**, and neighboring slices look almost identical. If slices are split randomly, the test set contains near-copies of training slices. The correct protocol is to split **by patient**. Experiment E measures the difference.

### Experiment design

| ID | Skip connections | Loss | Augmentation | Split |
|----|:---------------:|------|:------------:|-------|
| A | no | BCE + Dice | no | by patient |
| B | yes | BCE | no | by patient |
| C | yes | BCE + Dice | no | by patient |
| D | yes | BCE + Dice | yes | by patient |
| E | yes | BCE + Dice | no | **random slices (leaky)** |

C is the reference. A vs C isolates the skip connections, B vs C the loss, C vs D augmentation, and C vs E the leakage. Everything else is identical (architecture width, optimizer, schedule, epochs, seed).

In [ ]:
import os, copy, math, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
EPOCHS = 25
BATCH_SIZE = 16
LR = 1e-3
BASE = 32            # channels of the first U-Net level (32 -> 64 -> 128 -> 256 -> 512)
THRESH = 0.5         # probability threshold for the predicted mask
MIN_AREA = 10        # a slice counts as "tumor detected" if the predicted mask has >= 10 pixels

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(exist_ok=True)

def save_fig(name):
    plt.savefig(OUT / name, dpi=150, bbox_inches="tight")

print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 2. Dataset

**LGG MRI Segmentation** (Mateusz Buda, Kaggle): brain MRI slices of patients with lower-grade glioma, from The Cancer Genome Atlas, with manually drawn FLAIR-abnormality masks. Each case is a folder; each slice is an image `<case>_<n>.tif` (3 channels: pre-contrast, FLAIR, post-contrast) with a mask `<case>_<n>_mask.tif`. About 110 patients and 3,929 slice pairs of size 256×256.

The code below searches `/kaggle/input` for all `*_mask.tif` files and pairs each with its image, so it works whatever the exact folder layout of the added dataset is (duplicate nested copies are removed).

**Data pipeline.** The whole dataset is small (about 1 GB as 8-bit arrays), so I load it **once into memory and keep it on the GPU**. Batches are sliced directly from GPU tensors and augmentation is done on the GPU. This avoids DataLoader worker processes entirely, which is faster and removes a common source of crashes.

**Split.** 70% / 15% / 15% of the **patients** (all slices of a patient go to the same split). Patients are identified by their folder name.

In [ ]:
def list_input(root="/kaggle/input", max_items=12):
    root = Path(root)
    if not root.exists():
        print(root, "does not exist (are you running on Kaggle?)"); return
    items = sorted(root.iterdir())
    print(f"{root} contains {len(items)} item(s)")
    for p in items[:max_items]:
        print("  ", p)
        if p.is_dir():
            for q in sorted(p.iterdir())[:max_items]:
                print("      ", q)

def describe_input(root="/kaggle/input", n_examples=6):
    """Count the files under /kaggle/input by extension and show a few examples."""
    from collections import Counter
    files = [p for p in Path(root).rglob("*") if p.is_file()] if Path(root).exists() else []
    print(f"{len(files):,} files under {root}")
    by_ext = Counter(p.suffix.lower() for p in files)
    for ext, c in by_ext.most_common(8):
        print(f"  {ext or '(no extension)'}: {c:,}")
    for p in files[:n_examples]:
        print("   e.g.", p)

def find_pairs(root="/kaggle/input"):
    """Return a list of (image_path, mask_path, patient_id), duplicates removed.
    A mask is any image file whose name ends with '_mask' (case-insensitive); its image has the same name without it."""
    import re
    exts = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}
    pairs = {}
    for m in sorted(Path(root).rglob("*")):
        if m.suffix.lower() in exts and re.search(r"_mask$", m.stem, flags=re.I):
            img = m.with_name(re.sub(r"_mask$", "", m.stem, flags=re.I) + m.suffix)
            if img.exists() and m.name not in pairs:
                pairs[m.name] = (img, m, m.parent.name)
    if not pairs:
        list_input(root)
        describe_input(root)
        raise FileNotFoundError("No '*_mask' image files found under /kaggle/input (see the listing above). "
                                "If it is empty, add the 'LGG MRI Segmentation' dataset with '+ Add Input' and re-run.")
    return list(pairs.values())

pairs = find_pairs()
patients_all = sorted({p[2] for p in pairs})
print(f"{len(pairs):,} image/mask pairs from {len(patients_all)} patients")
im0 = Image.open(pairs[0][0]); mk0 = Image.open(pairs[0][1])
print("first image:", im0.mode, im0.size, "| first mask:", mk0.mode, mk0.size)

In [ ]:
def load_all(pairs):
    n = len(pairs)
    X = np.zeros((n, 3, 256, 256), np.uint8)
    Y = np.zeros((n, 1, 256, 256), np.uint8)
    t0 = time.time()
    for i, (ip, mp, _) in enumerate(pairs):
        im = Image.open(ip).convert("RGB"); mk = Image.open(mp).convert("L")
        if im.size != (256, 256):
            im = im.resize((256, 256), Image.BILINEAR); mk = mk.resize((256, 256), Image.NEAREST)
        X[i] = np.asarray(im).transpose(2, 0, 1)
        Y[i, 0] = np.asarray(mk) > 127
        if (i + 1) % 1000 == 0:
            print(f"  loaded {i + 1}/{n} ({time.time() - t0:.0f}s)")
    return X, Y

X_np, Y_np = load_all(pairs)
patient_of = np.array([p[2] for p in pairs])
areas = Y_np.reshape(len(Y_np), -1).sum(1)
has_tumor = areas > 0
print(f"slices with a tumor: {has_tumor.sum():,} ({has_tumor.mean():.1%}) | "
      f"tumor pixels overall: {Y_np.mean():.2%} of all pixels, {Y_np[has_tumor].mean():.2%} within tumor slices")

X = torch.from_numpy(X_np).to(DEVICE)      # uint8 on the GPU
Y = torch.from_numpy(Y_np).to(DEVICE)

In [ ]:
# ---- patient-level split (the correct protocol) ----
rng = np.random.RandomState(SEED)
pl = list(patients_all); rng.shuffle(pl)
n_test = int(round(0.15 * len(pl))); n_val = int(round(0.15 * len(pl)))
test_p, val_p, train_p = set(pl[:n_test]), set(pl[n_test:n_test + n_val]), set(pl[n_test + n_val:])
by_patient = tuple(np.where(np.isin(patient_of, list(s)))[0] for s in (train_p, val_p, test_p))

# ---- random slice-level split (leaky; used only in experiment E) ----
perm = np.random.RandomState(SEED).permutation(len(pairs))
n_tr = int(0.70 * len(pairs)); n_va = int(0.15 * len(pairs))
by_slice = (np.sort(perm[:n_tr]), np.sort(perm[n_tr:n_tr + n_va]), np.sort(perm[n_tr + n_va:]))

SPLITS = {"patient": by_patient, "slice": by_slice}

rows = []
for sname, (tr, va, te) in SPLITS.items():
    for part, idx in zip(("train", "val", "test"), (tr, va, te)):
        rows.append({"split": sname, "part": part, "slices": len(idx), "patients": len(set(patient_of[idx])),
                     "tumor slices (%)": round(100 * has_tumor[idx].mean(), 1)})
split_table = pd.DataFrame(rows)
display(split_table) if "display" in globals() else print(split_table)

for sname, (tr, va, te) in SPLITS.items():
    shared = set(patient_of[tr]) & set(patient_of[te])
    print(f"{sname:>7} split: {len(shared)} patients appear in BOTH train and test")

In [ ]:
# Normalization statistics from the (patient-level) training slices only
tr_idx0 = torch.as_tensor(by_patient[0], device=DEVICE)
s1 = torch.zeros(3, device=DEVICE, dtype=torch.float64); s2 = torch.zeros(3, device=DEVICE, dtype=torch.float64)
cnt = 0
for i in range(0, len(tr_idx0), 256):
    xb = X[tr_idx0[i:i + 256]].double() / 255.0
    s1 += xb.sum((0, 2, 3)); s2 += (xb ** 2).sum((0, 2, 3)); cnt += xb.numel() // 3
MEAN = (s1 / cnt).float().view(1, 3, 1, 1)
STD = ((s2 / cnt - (s1 / cnt) ** 2).sqrt()).float().view(1, 3, 1, 1)
print("channel mean:", MEAN.flatten().tolist(), "| std:", STD.flatten().tolist())

def prep(xb_uint8):
    return (xb_uint8.float() / 255.0 - MEAN) / STD

In [ ]:
# Look at the data: samples with ground-truth contours, and the tumor-size distribution
def show(ax, i, mask=None, color="lime"):
    ax.imshow(X_np[i].transpose(1, 2, 0)); ax.axis("off")
    m = Y_np[i, 0] if mask is None else mask
    if m.any():
        ax.contour(m, levels=[0.5], colors=color, linewidths=1.2)

rs = np.random.RandomState(SEED)
pos_ids = rs.choice(np.where(has_tumor)[0], 6, replace=False)
neg_ids = rs.choice(np.where(~has_tumor)[0], 6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(15, 5.2))
for ax, i in zip(axes[0], pos_ids):
    show(ax, i); ax.set_title(f"tumor, {areas[i]} px", fontsize=9)
for ax, i in zip(axes[1], neg_ids):
    show(ax, i); ax.set_title("no tumor", fontsize=9)
plt.suptitle("Brain MRI slices (R/G/B = pre-contrast / FLAIR / post-contrast), ground truth in green")
plt.tight_layout()
save_fig("data_samples.png")
plt.show()

plt.figure(figsize=(7, 3.5))
plt.hist(areas[has_tumor], bins=np.logspace(np.log10(max(1, areas[has_tumor].min())), np.log10(areas.max()), 40), color="tab:green")
plt.xscale("log"); plt.xlabel("tumor area in a slice (pixels, log scale)"); plt.ylabel("slices")
plt.title("Tumor size distribution (tumor slices only)")
plt.tight_layout()
save_fig("tumor_size_distribution.png")
plt.show()

## 3. Model Implementation

`UNet(skip=True)` is the standard U-Net. With `skip=False` the concatenation is removed and the decoder receives only the upsampled features, which gives the **same encoder-decoder without skip connections** (a bit fewer parameters, because the decoder blocks take half as many input channels).

| Stage | Output (256×256 input) |
|-------|------------------------|
| Encoder 1: DoubleConv 3→32 | 32×256×256 |
| Encoder 2: pool + DoubleConv 32→64 | 64×128×128 |
| Encoder 3: pool + DoubleConv 64→128 | 128×64×64 |
| Encoder 4: pool + DoubleConv 128→256 | 256×32×32 |
| Bottleneck: pool + DoubleConv 256→512 | 512×16×16 |
| Decoder 4: up 512→256, concat skip, DoubleConv | 256×32×32 |
| Decoder 3 / 2 / 1 | 128×64×64 / 64×128×128 / 32×256×256 |
| Head: Conv 1×1 | 1×256×256 (logits) |

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=1, base=32, skip=True):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]
        self.skip = skip
        self.enc = nn.ModuleList([DoubleConv(in_ch, c[0]), DoubleConv(c[0], c[1]),
                                  DoubleConv(c[1], c[2]), DoubleConv(c[2], c[3])])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[4])
        self.up = nn.ModuleList([nn.ConvTranspose2d(c[i + 1], c[i], 2, stride=2) for i in reversed(range(4))])
        self.dec = nn.ModuleList([DoubleConv(c[i] * (2 if skip else 1), c[i]) for i in reversed(range(4))])
        self.head = nn.Conv2d(c[0], n_classes, 1)

    def forward(self, x):
        feats = []
        for blk in self.enc:
            x = blk(x); feats.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for up, dec, f in zip(self.up, self.dec, reversed(feats)):
            x = up(x)
            if self.skip:
                x = torch.cat([f, x], dim=1)     # skip connection: concatenate encoder features
            x = dec(x)
        return self.head(x)


def count_params(m):
    return sum(p.numel() for p in m.parameters())

for sk, expected in [(True, 7_763_041), (False, 6_979_681)]:
    n = count_params(UNet(base=BASE, skip=sk))
    note = f"matches the hand-computed {expected:,}" if (BASE == 32 and n == expected) else "(hand-computed value is for BASE=32)"
    print(f"U-Net base {BASE}, skip={sk}: {n:,} parameters {note}")

x = torch.randn(2, 3, 256, 256)
print("output shape:", tuple(UNet(base=BASE)(x).shape))

## 4. Training Setup

- **Loss:** BCE with logits, or BCE + soft Dice (batch-level) | **Optimizer:** Adam, lr = 1e-3 | **Schedule:** cosine annealing over 25 epochs | batch size 16 | mixed precision
- **Model selection:** the epoch with the best **validation global Dice**; the test set is evaluated once per run
- **Threshold:** 0.5 on the sigmoid output
- **Augmentation (experiment D only):** done on the GPU with one random affine transform per slice: horizontal flip, rotation within ±15°, scaling within ±10%, translation within ±10%. The *same* transform is applied to the image (bilinear) and to the mask (nearest neighbor).

In [ ]:
use_amp = DEVICE.type == "cuda"

def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum()
    return 1 - (2 * inter + eps) / (p.sum() + target.sum() + eps)

def compute_loss(logits, target, kind):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    return bce if kind == "bce" else bce + dice_loss(logits, target)

def augment(x, y):
    """Random affine on a GPU batch: same transform for image (bilinear) and mask (nearest)."""
    B, dev = x.size(0), x.device
    u = lambda lo, hi: lo + (hi - lo) * torch.rand(B, device=dev)
    ang, sc = u(-15, 15) * math.pi / 180, u(0.9, 1.1)
    tx, ty = u(-0.1, 0.1), u(-0.1, 0.1)
    flip = (torch.rand(B, device=dev) < 0.5).float() * 2 - 1
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = cos * flip; theta[:, 0, 1] = -sin; theta[:, 0, 2] = tx
    theta[:, 1, 0] = sin * flip; theta[:, 1, 1] = cos;  theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    y = F.grid_sample(y, grid, mode="nearest", padding_mode="zeros", align_corners=False)
    return x, (y > 0.5).float()

@torch.no_grad()
def predict(model, idx, bs=64):
    """Probabilities (N,1,H,W, float16 on device) and mean BCE for the given slice indices."""
    model.eval()
    idx_t = torch.as_tensor(idx, device=DEVICE)
    probs, bce_sum = [], 0.0
    for i in range(0, len(idx_t), bs):
        j = idx_t[i:i + bs]
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            logits = model(prep(X[j]))
        logits = logits.float()
        bce_sum += F.binary_cross_entropy_with_logits(logits, Y[j].float(), reduction="sum").item()
        probs.append(torch.sigmoid(logits).half())
    probs = torch.cat(probs)
    return probs, bce_sum / (len(idx_t) * 256 * 256)

def slice_stats(probs, idx):
    pred = (probs > THRESH).float()
    g = Y[torch.as_tensor(idx, device=DEVICE)].float()
    inter = (pred * g).sum((1, 2, 3)).double().cpu().numpy()
    p = pred.sum((1, 2, 3)).double().cpu().numpy()
    gs = g.sum((1, 2, 3)).double().cpu().numpy()
    return inter, p, gs

def summarize(inter, p, g):
    pos = g > 0
    dice_slice = 2 * inter / (p + g + 1e-8)
    iou_slice = inter / (p + g - inter + 1e-8)
    return {
        "dice": 2 * inter.sum() / (p.sum() + g.sum() + 1e-8),
        "iou": inter.sum() / (p.sum() + g.sum() - inter.sum() + 1e-8),
        "dice_tumor": dice_slice[pos].mean() if pos.any() else float("nan"),
        "iou_tumor": iou_slice[pos].mean() if pos.any() else float("nan"),
        "sensitivity": (p[pos] >= MIN_AREA).mean() if pos.any() else float("nan"),
        "specificity": (p[~pos] < MIN_AREA).mean() if (~pos).any() else float("nan"),
    }

def train_experiment(name, skip, loss_kind, augment_on, split):
    print(f"\n=== {name} | skip={skip} loss={loss_kind} aug={augment_on} split={split} ===")
    torch.manual_seed(SEED)
    tr_idx, va_idx, te_idx = SPLITS[split]
    tr_t = torch.as_tensor(tr_idx, device=DEVICE)

    model = UNet(base=BASE, skip=skip).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    hist = {"train_loss": [], "val_bce": [], "val_dice": []}
    best_dice, best_epoch, best_state = -1.0, 0, None
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train()
        order = tr_t[torch.randperm(len(tr_t), device=DEVICE)]
        total, nb = 0.0, 0
        for i in range(0, len(order), BATCH_SIZE):
            j = order[i:i + BATCH_SIZE]
            if len(j) < 2:
                continue
            xb, yb = prep(X[j]), Y[j].float()
            if augment_on:
                xb, yb = augment(xb, yb)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits = model(xb)
            loss = compute_loss(logits.float(), yb, loss_kind)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total += loss.item(); nb += 1
        scheduler.step()

        probs, vbce = predict(model, va_idx)
        vd = summarize(*slice_stats(probs, va_idx))["dice"]
        hist["train_loss"].append(total / nb); hist["val_bce"].append(vbce); hist["val_dice"].append(vd)
        if vd > best_dice:
            best_dice, best_epoch = vd, ep
            best_state = copy.deepcopy(model.state_dict())
        print(f"Epoch {ep:02d}/{EPOCHS} | train loss {total / nb:.4f} | val BCE {vbce:.4f} | val Dice {vd:.4f} | {time.time() - t0:.0f}s")
    elapsed = time.time() - t0

    model.load_state_dict(best_state)
    probs, _ = predict(model, te_idx)
    inter, p, g = slice_stats(probs, te_idx)
    metrics = summarize(inter, p, g)
    print(f"-> best epoch {best_epoch} | val Dice {best_dice:.4f} | test Dice {metrics['dice']:.4f} | "
          f"test IoU {metrics['iou']:.4f} | tumor-slice Dice {metrics['dice_tumor']:.4f} | {elapsed:.0f}s")
    return dict(model=model, hist=hist, skip=skip, loss=loss_kind, aug=augment_on, split=split,
                params=count_params(model), best_epoch=best_epoch, best_val_dice=best_dice,
                metrics=metrics, per_slice=(inter, p, g), te_idx=te_idx, time=elapsed)

results = {}

### Experiment A: no skips (BCE+Dice)

In [ ]:
results["A: no skips (BCE+Dice)"] = train_experiment("A: no skips (BCE+Dice)", skip=False, loss_kind="bce_dice", augment_on=False, split="patient")

### Experiment B: U-Net (BCE only)

In [ ]:
results["B: U-Net (BCE only)"] = train_experiment("B: U-Net (BCE only)", skip=True, loss_kind="bce", augment_on=False, split="patient")

### Experiment C: U-Net (BCE+Dice)

In [ ]:
results["C: U-Net (BCE+Dice)"] = train_experiment("C: U-Net (BCE+Dice)", skip=True, loss_kind="bce_dice", augment_on=False, split="patient")

### Experiment D: U-Net + augmentation

In [ ]:
results["D: U-Net + augmentation"] = train_experiment("D: U-Net + augmentation", skip=True, loss_kind="bce_dice", augment_on=True, split="patient")

### Experiment E: slice-level split (leaky)

In [ ]:
results["E: slice-level split (leaky)"] = train_experiment("E: slice-level split (leaky)", skip=True, loss_kind="bce_dice", augment_on=False, split="slice")

## 5. Results

In [ ]:
rows = []
for name, r in results.items():
    m = r["metrics"]
    rows.append({
        "Config": name,
        "Skips": "yes" if r["skip"] else "no",
        "Loss": r["loss"],
        "Aug": "yes" if r["aug"] else "no",
        "Split": r["split"],
        "Params": f"{r['params']:,}",
        "Best epoch": r["best_epoch"],
        "Val Dice": round(r["best_val_dice"], 4),
        "Test Dice": round(m["dice"], 4),
        "Test IoU": round(m["iou"], 4),
        "Dice (tumor slices)": round(m["dice_tumor"], 4),
        "Sensitivity": round(m["sensitivity"], 3),
        "Specificity": round(m["specificity"], 3),
        "Time (s)": round(r["time"]),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "results_summary.csv", index=False)
summary

In [ ]:
# Training curves
names = list(results)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
for n in names:
    h = results[n]["hist"]; ep = np.arange(1, len(h["val_dice"]) + 1)
    ax[0].plot(ep, h["val_dice"], label=n)
    ax[1].plot(ep, h["train_loss"], label=n)
    ax[2].plot(ep, h["val_bce"], label=n)
ax[0].set_title("Validation Dice (global)"); ax[0].set_ylim(0.4, 1.0)
ax[1].set_title("Training loss (each run's own loss)")
ax[2].set_title("Validation BCE")
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3)
ax[0].legend(fontsize=7)
plt.tight_layout()
save_fig("training_curves_all.png")
plt.show()

In [ ]:
# Comparison of the patient-level experiments (A to D)
pat = [n for n in names if results[n]["split"] == "patient"]
metrics_to_plot = [("dice", "Global Dice"), ("dice_tumor", "Dice (tumor slices)"), ("sensitivity", "Sensitivity"), ("specificity", "Specificity")]
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, (k, title) in zip(ax, metrics_to_plot):
    vals = [results[n]["metrics"][k] for n in pat]
    bars = a.bar([n.split(":")[0] for n in pat], vals, color=["tab:gray", "tab:orange", "tab:blue", "tab:green"][:len(pat)])
    for b, v in zip(bars, vals):
        a.text(b.get_x() + b.get_width() / 2, v + 0.003, f"{v:.3f}", ha="center", fontsize=9)
    a.set_ylim(max(0, min(vals) - 0.1), min(1.02, max(vals) + 0.06)); a.set_title(title)
plt.suptitle("Experiments A to D (patient-level split, same test patients)")
plt.tight_layout()
save_fig("comparison_AtoD.png")
plt.show()

## Qualitative Results and Error Analysis

The best configuration among the patient-split experiments is selected by **validation Dice**.

In [ ]:
best_name = max(pat, key=lambda k: results[k]["best_val_dice"])
best = results[best_name]
te_idx = best["te_idx"]
inter, p, g = best["per_slice"]
pos = g > 0
dice_slice = 2 * inter / (p + g + 1e-8)
print("Best config by validation Dice:", best_name,
      f"| test Dice {best['metrics']['dice']:.4f} | IoU {best['metrics']['iou']:.4f}")

probs_best, _ = predict(best["model"], te_idx)
pred_masks = (probs_best > THRESH).squeeze(1).cpu().numpy()

rs = np.random.RandomState(SEED)
pos_pos = np.where(pos)[0]
random_ids = rs.choice(pos_pos, 4, replace=False)
worst_ids = pos_pos[np.argsort(dice_slice[pos_pos])[:4]]
ids = list(random_ids) + list(worst_ids)

fig, axes = plt.subplots(3, 8, figsize=(19, 7.2))
for c, k in enumerate(ids):
    gi = te_idx[k]
    axes[0, c].imshow(X_np[gi].transpose(1, 2, 0)); axes[0, c].axis("off")
    axes[0, c].set_title(("random" if c < 4 else "worst") + f" | Dice {dice_slice[k]:.2f}", fontsize=9)
    axes[1, c].imshow(X_np[gi].transpose(1, 2, 0)); axes[1, c].axis("off")
    axes[1, c].contour(Y_np[gi, 0], levels=[0.5], colors="lime", linewidths=1.2)
    axes[2, c].imshow(X_np[gi].transpose(1, 2, 0)); axes[2, c].axis("off")
    if pred_masks[k].any():
        axes[2, c].contour(pred_masks[k], levels=[0.5], colors="red", linewidths=1.2)
axes[0, 0].set_ylabel("image"); 
fig.text(0.005, 0.80, "image", rotation=90, fontsize=11); fig.text(0.005, 0.50, "ground truth", rotation=90, fontsize=11)
fig.text(0.005, 0.17, "prediction", rotation=90, fontsize=11)
plt.suptitle(f"{best_name}: 4 random tumor slices (left) and the 4 worst tumor slices (right)")
plt.tight_layout()
save_fig("predictions_best.png")
plt.show()

In [ ]:
# Dice vs tumor size (tumor slices of the best model)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].scatter(g[pos], dice_slice[pos], s=10, alpha=0.5)
ax[0].set_xscale("log"); ax[0].set_xlabel("ground-truth tumor area (pixels, log scale)"); ax[0].set_ylabel("per-slice Dice")
ax[0].set_title("Small tumors are harder"); ax[0].grid(alpha=0.3)
ax[1].hist(dice_slice[pos], bins=25, color="tab:blue")
ax[1].set_xlabel("per-slice Dice (tumor slices)"); ax[1].set_ylabel("slices")
ax[1].set_title(f"Per-slice Dice distribution (median {np.median(dice_slice[pos]):.3f})")
plt.tight_layout()
save_fig("dice_vs_size.png")
plt.show()

small = pos & (g < np.median(g[pos])); large = pos & (g >= np.median(g[pos]))
print(f"mean Dice: smaller half of the tumors {dice_slice[small].mean():.3f} | larger half {dice_slice[large].mean():.3f}")
print(f"tumor slices with Dice < 0.5: {(dice_slice[pos] < 0.5).mean():.1%} | missed completely (no prediction): {(p[pos] < MIN_AREA).mean():.1%}")
print(f"slices without a tumor with a false alarm (>= {MIN_AREA} px): {(p[~pos] >= MIN_AREA).mean():.1%}")

In [ ]:
# Effect of the skip connections: same slices, model A (no skips) vs model C (with skips)
rsq = np.random.RandomState(SEED + 1)
a_res, c_res = results["A: no skips (BCE+Dice)"], results["C: U-Net (BCE+Dice)"]
te_idx_ac = c_res["te_idx"]
inter_c, p_c, g_c = c_res["per_slice"]; inter_a, p_a, g_a = a_res["per_slice"]
pos_ac = np.where(g_c > 0)[0]
sel = rsq.choice(pos_ac, 6, replace=False)
pa, _ = predict(a_res["model"], te_idx_ac[sel]); pc, _ = predict(c_res["model"], te_idx_ac[sel])
ma, mc = (pa > THRESH).squeeze(1).cpu().numpy(), (pc > THRESH).squeeze(1).cpu().numpy()

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for c, k in enumerate(sel):
    gi = te_idx_ac[k]
    for r in range(3):
        axes[r, c].imshow(X_np[gi].transpose(1, 2, 0)); axes[r, c].axis("off")
    axes[0, c].contour(Y_np[gi, 0], levels=[0.5], colors="lime", linewidths=1.2); axes[0, c].set_title("ground truth", fontsize=9)
    if ma[c].any(): axes[1, c].contour(ma[c], levels=[0.5], colors="red", linewidths=1.2)
    axes[1, c].set_title(f"A, no skips | Dice {2 * (ma[c] & Y_np[gi, 0].astype(bool)).sum() / (ma[c].sum() + g_c[k] + 1e-8):.2f}", fontsize=9)
    if mc[c].any(): axes[2, c].contour(mc[c], levels=[0.5], colors="red", linewidths=1.2)
    axes[2, c].set_title(f"C, with skips | Dice {2 * (mc[c] & Y_np[gi, 0].astype(bool)).sum() / (mc[c].sum() + g_c[k] + 1e-8):.2f}", fontsize=9)
plt.suptitle("Skip connections: ground truth (top), no skips (middle), U-Net (bottom)")
plt.tight_layout()
save_fig("skip_comparison.png")
plt.show()

## The Leakage Experiment (C vs E)

Experiment E has the same model and settings as C but uses a **random slice-level split**. Each model is evaluated on **its own test set** (the two test sets contain different slices), so the comparison is indicative, not a controlled test of the same images. The key difference is that in E most test patients also appear in the training set.

In [ ]:
c_m, e_m = results["C: U-Net (BCE+Dice)"]["metrics"], results["E: slice-level split (leaky)"]["metrics"]
keys = [("dice", "Global Dice"), ("iou", "Global IoU"), ("dice_tumor", "Dice (tumor slices)"), ("sensitivity", "Sensitivity")]
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(keys)); w = 0.36
b1 = ax.bar(x - w / 2, [c_m[k] for k, _ in keys], w, label="C: split by patient", color="tab:blue")
b2 = ax.bar(x + w / 2, [e_m[k] for k, _ in keys], w, label="E: random slices (leaky)", color="tab:red")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.004, f"{b.get_height():.3f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([t for _, t in keys])
lo = min(min(c_m[k], e_m[k]) for k, _ in keys); ax.set_ylim(max(0, lo - 0.1), 1.02)
ax.set_title("Same model and training, different split protocol"); ax.legend()
plt.tight_layout()
save_fig("leakage_comparison.png")
plt.show()

for k, t in keys:
    print(f"{t:<22} patient split {c_m[k]:.4f} | slice split {e_m[k]:.4f} | difference {e_m[k] - c_m[k]:+.4f}")

In [ ]:
# Save the best model
torch.save(best["model"].state_dict(), OUT / "unet_best.pth")
print("Saved to", OUT)

## 6. Analysis

### Key findings

**1. Overall performance.** The best patient-split configuration, selected on validation, is B (U-Net trained with BCE only): test Dice 0.754, IoU 0.605, mean Dice on tumor slices 0.641, sensitivity 0.857 and specificity 0.924.

**2. Data leakage inflates the scores a lot.** With the same model and training (C vs E), a random slice-level split instead of a patient-level split raises global Dice from 0.673 to 0.836 (+0.163), IoU from 0.507 to 0.718 (+0.211), Dice on tumor slices from 0.519 to 0.725 (+0.206), sensitivity from 0.753 to 0.882 and specificity from 0.905 to 0.959. Validation Dice is also higher in E (0.827 vs 0.732) because its validation split is leaky as well. A plausible explanation is that neighboring slices of the same patient are near-copies, so the model can recognize "this brain" instead of learning what tumors look like. The leakage effect (+0.16 Dice) is about twice the whole spread between configurations A to D (0.666 to 0.754), so for this setup the evaluation protocol mattered more than any architecture, loss or augmentation choice I tested. Caveat: the two test sets contain different slices, and this is a single seed.

**3. Skip connections: no clear benefit here.** A (no skips) vs C (skips): Dice 0.666 vs 0.673, IoU 0.499 vs 0.507, Dice on tumor slices 0.502 vs 0.519, sensitivity 0.731 vs 0.753, validation Dice 0.714 vs 0.732. Every number favors the skips, but the differences are small and within the noise level of this experiment (see point 9). The six-slice qualitative comparison is mixed: the U-Net is better on one slice (Dice 0.31 vs 0.00), the no-skip model is better on another (0.72 vs 0.14), they are similar on a third, and both completely miss the other three. I expected a clearer advantage in boundary quality. Possible reasons (hypotheses, not tested): the tumors in this dataset are fairly large blob-like regions where coarse localization matters more than fine boundaries; the dominant error is failing to detect the tumor in new patients, not boundary precision; and training is short with few patients. The skips cost about 0.78M more parameters (+11%) and about 11% more training time (735 s vs 662 s).

**4. Loss: BCE alone beat BCE + Dice (unexpectedly).** B vs C: test Dice 0.754 vs 0.673 (+0.081), IoU 0.605 vs 0.507, Dice on tumor slices 0.641 vs 0.519, sensitivity 0.857 vs 0.753, specificity 0.924 vs 0.905. I expected the Dice term to help with class imbalance. Two cautions: on the validation set the gap is only 0.017 (0.748 vs 0.732), so the large test gap is probably inflated by the small test set; and validation Dice fluctuates by several hundredths between epochs in all patient-split runs. Possible reasons for BCE doing well (not tested): BCE may be sufficient at this level of imbalance, and the batch-level Dice term can be noisy when most slices in a batch are empty. I cannot rank BCE and BCE + Dice with confidence from one seed, but there is no evidence here that adding Dice helps.

**5. Augmentation: small gains on test, not on validation.** C vs D: Dice 0.673 → 0.691, IoU 0.507 → 0.527, Dice on tumor slices 0.519 → 0.571, sensitivity 0.753 → 0.775, specificity 0.905 → 0.896, validation Dice 0.732 → 0.736. The direction is positive on most metrics, but the size is small compared with the noise, so I read it as suggestive only. The training loss ends higher with augmentation (about 0.22 vs 0.19), as expected for harder training data, and GPU-side augmentation added no measurable time (739 s vs 735 s).

**6. The main failure mode is missed tumors.** For the best model, the median per-slice Dice on tumor slices is 0.759 but the mean is 0.641, because the distribution is bimodal: a spike at Dice 0 (14.3% of tumor slices are missed completely, consistent with the sensitivity of 0.857) and most other slices between 0.6 and 0.95. 23.6% of tumor slices have Dice below 0.5. If the completely missed slices were excluded, the mean Dice would be about 0.75 (0.641 / 0.857, my approximation). So when the model finds a tumor it usually segments it well (the four random examples have Dice 0.84 to 0.95); the dominant error is not seeing the tumor at all. On the other side, 7.6% of slices without a tumor get a false alarm (at least 10 predicted pixels).

**7. Tumor size matters on average, but it is not the whole story.** The smaller half of the tumors has a mean Dice of 0.512 vs 0.770 for the larger half. However, several complete misses have areas of about 1,000 to 3,000 pixels, so large tumors are also missed. Of the four worst slices (all Dice 0.00), one is a tiny lesion and three are moderate lesions that are faint in the image; three of the four look like they come from the same scan (similar appearance and brain level), which is a hypothesis from visual inspection.

**8. Scans are heterogeneous.** The slices differ strongly in appearance across patients: some have a dark red background, others a green or blue tint, which probably reflects different acquisition protocols or channel intensity scales. This is a plausible contributor to the cross-patient generalization gap, but I did not test it (for example with per-image intensity normalization). In the skip-connection figure, three of the six random slices (same red background and similar anatomy) are missed or poorly segmented by both models, which again suggests that errors cluster by patient.

**9. Noisy training and model selection.** Validation Dice fluctuates by several hundredths between consecutive epochs in the patient-split runs, and the spread across A to D on validation (0.714 to 0.748) is of the same order. The best epochs are 18 to 21 out of 25. With only 16 validation and about 17 test patients, slices from the same patient are not independent, so uncertainty is larger than the number of slices suggests; a bootstrap over patients and several seeds would be needed to rank configurations reliably.

### Limitations
- Single run per configuration (one seed). The differences among A to D (about 0.09 in test Dice) are of the same order as the epoch-to-epoch noise in validation Dice, so none of the A to D rankings is conclusive.
- Only about 110 patients, with about 17 in the test set; slices of one patient are correlated, and no confidence intervals were computed.
- The comparison between C and E uses two different test sets.
- The masks are a single annotation of FLAIR abnormality from one public collection, with no external validation; slices are very heterogeneous across patients.
- 25 epochs, a small U-Net (base 32), a fixed threshold of 0.5 and no post-processing or hyperparameter search. Some runs may be undertrained.

### Possible improvements
- Several seeds and bootstrap confidence intervals over patients; patient-level cross-validation
- Per-image intensity normalization or per-channel standardization to reduce scan heterogeneity
- Post-processing (remove tiny predicted components to cut false alarms, threshold tuning on validation)
- Test other Dice formulations (per-image Dice, focal Tversky), pretrained encoders, Attention U-Net, 3D context from neighboring slices
- Evaluate on an external dataset

### Next in this series
**Day 6:** LSTM/GRU for sentiment analysis (sequence modeling).

### References
- Ronneberger, O., Fischer, P., Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.
- Buda, M., Saha, A., Mazurowski, M. A. (2019). *Association of genomic subtypes of lower-grade gliomas with shape features automatically extracted by a deep learning algorithm.* Computers in Biology and Medicine, 109.
- Mazurowski, M. A. et al. (2017). *Radiogenomics of lower-grade glioma: algorithmically-assessed tumor shape is associated with tumor genomic subtypes and patient outcomes in a multi-institutional study with The Cancer Genome Atlas data.* Journal of Neuro-Oncology.
- Milletari, F., Navab, N., Ahmadi, S.-A. (2016). *V-Net: Fully Convolutional Neural Networks for Volumetric Medical Image Segmentation.* 3DV (soft Dice loss).